# Assignment 2: Learning from Click-Logs — Improved NRMS Pipeline
## CS4.406 — Information Retrieval & Extraction

**Architecture: 3 baselines × Improved NRMS re-ranker × 2 datasets**

### Improvements over baseline notebook:
1. **Larger training data** — train on full MINDsmall_train (not dev); use ebnerd_large when available
2. **Richer embeddings** — use raw 384-dim MiniLM (MIND) / 300-dim BERT (EB-NeRD) directly as news representations
3. **More negatives** — npratio=8 instead of 4 for better listwise discrimination
4. **Larger model** — d_model projected to 256, 8 attention heads, 2-layer news encoder with residual
5. **Longer user history** — 50 articles instead of 30
6. **LR scheduling** — linear warmup + cosine decay

| Section | What |
|---------|------|
| 0 | Setup, config, A1 data loading |
| 1 | Feature engineering (Q1) |
| 2 | Three baselines (B1, B2, B3) |
| 3 | Improved NRMS re-ranker (Q2) |
| 4 | Full pipeline run (Q3) |
| 5 | Serving & scale analysis (Q4) |
| 6 | Codabench submissions |
| 7 | Anti-leakage tests (Q9) |

In [1]:
# %pip install polars pyarrow numpy faiss-cpu rank-bm25 nltk scikit-learn tqdm matplotlib torch lightgbm
# %pip install sentence-transformers

import polars as pl, pyarrow.parquet as pq, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass, field
from tqdm.auto import tqdm
from functools import lru_cache
from datetime import datetime
from sklearn.metrics import roc_auc_score
from rank_bm25 import BM25Okapi
import json, gc, zipfile, re, os, time, sys, math

import nltk
for r in ["punkt","punkt_tab","stopwords"]:
    try: nltk.data.find(f"tokenizers/{r}" if "punkt" in r else f"corpora/{r}")
    except LookupError: nltk.download(r, quiet=True)
from nltk.corpus import stopwords as _sw
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.tokenize import word_tokenize

try: import faiss; HAS_FAISS=True
except: HAS_FAISS=False
import torch
import torch.nn as nn
import torch.nn.functional as F
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

np.random.seed(42); torch.manual_seed(42)
print(f"Polars {pl.__version__} | FAISS={HAS_FAISS} | Device={DEVICE} | GPU={torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'none'}")
import psutil


Polars 1.44.0 | FAISS=True | Device=cuda | GPU=NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
# ============================================================
# CONFIGURATION — IMPROVED HYPERPARAMETERS
# ============================================================
@dataclass
class Cfg:
    ebnerd_demo: Path = Path("data/raw/ebnerd_demo")
    ebnerd_large: Path = Path("data/raw/ebnerd_large")
    ebnerd_test: Path = Path("data/raw/ebnerd_testset")
    mind_small_train: Path = Path("data/raw/MINDsmall_train")
    mind_large_train: Path = Path("data/raw/MINDlarge_train/MINDlarge_train")
    mind_dev: Path = Path("data/raw/MINDsmall_dev")
    mind_test: Path = Path("data/raw/MINDlarge_test")
    out: Path = Path("data/processed")
    # ---- Hybrid training strategy ----
    # Load all of demo/small, then top up from large until total = multiplier * demo/small size.
    # This avoids OOM while giving 2-2.5x the training data of demo/small alone.
    train_multiplier: float = 2.25   # final size = multiplier * demo/small size
    # BM25
    k1: float = 1.5; b: float = 0.75; n_recent: int = 10
    # Embeddings
    embed_model: str = "all-MiniLM-L6-v2"; embed_bs: int = 256
    # Retrieval
    retrieve_k: int = 200
    # ---- IMPROVED NRMS hyperparameters ----
    nrms_proj_dim: int = 256       # project embeddings to this dim
    nrms_n_heads: int = 8          # attention heads
    nrms_hidden: int = 256         # feedforward hidden dim
    nrms_lr: float = 1e-3          # peak LR (with warmup+cosine decay)
    nrms_epochs: int = 8           # more epochs with scheduler
    nrms_batch: int = 128          # batch size
    nrms_hist_len: int = 50        # max history length
    nrms_n_neg: int = 8            # negatives per positive
    nrms_warmup_frac: float = 0.1  # fraction of steps for LR warmup
    nrms_dropout: float = 0.15     # dropout rate
    nrms_grad_clip: float = 1.0    # gradient clipping
    nrms_patience: int = 3         # early stopping patience
    # Eval
    cold_pctl: int = 25; boot_n: int = 1000; seed: int = 42
    ks: list = field(default_factory=lambda: [50, 100, 200])
    def __post_init__(self): self.out.mkdir(parents=True, exist_ok=True)
cfg = Cfg()
print(f"Config: proj_dim={cfg.nrms_proj_dim}, heads={cfg.nrms_n_heads}, hidden={cfg.nrms_hidden}, "
      f"hist={cfg.nrms_hist_len}, neg={cfg.nrms_n_neg}, epochs={cfg.nrms_epochs}, "
      f"warmup={cfg.nrms_warmup_frac}, dropout={cfg.nrms_dropout}, "
      f"train_multiplier={cfg.train_multiplier}x")


Config: proj_dim=256, heads=8, hidden=256, hist=50, neg=8, epochs=8, warmup=0.1, dropout=0.15, train_multiplier=2.25x


## 0. A1 Foundations (parsing, preprocessing, metrics)

In [3]:
# ============================================================
# TEXT PREPROCESSING + METRICS (from A1)
# ============================================================
@lru_cache(4)
def _stem(l): return PorterStemmer() if l=="english" else SnowballStemmer("danish")
@lru_cache(4)
def _stop(l):
    try: return frozenset(_sw.words(l))
    except: return frozenset()
def preprocess(text, lang="english"):
    if not text or not text.strip(): return []
    t=re.sub(r"&\w+;|<[^>]+>|[^\w\s]"," ",text.lower()); t=re.sub(r"\s+"," ",t).strip()
    toks=word_tokenize(t) if lang=="english" else t.split()
    st,sm=_stop(lang),_stem(lang)
    return [sm.stem(w) for w in toks if len(w)>1 and w not in st]

def recall_at_k(gt,ret,k):
    if not gt: return 0.
    top=[r[0] if isinstance(r,tuple) else r for r in ret[:k]]
    return len(set(gt)&set(top))/len(gt)
def mrr(labs,sc):
    if not labs: return 0.
    for r,i in enumerate(np.argsort(sc)[::-1],1):
        if labs[i]==1: return 1./r
    return 0.
def ndcg(labs,sc,k):
    if not labs: return 0.
    o=np.argsort(sc)[::-1]; dcg=sum(labs[o[i]]/np.log2(i+2) for i in range(min(k,len(labs))))
    ideal=sorted(labs,reverse=True); idcg=sum(ideal[i]/np.log2(i+2) for i in range(min(k,len(ideal))))
    return dcg/idcg if idcg>0 else 0.
def boot_ci(vals,n=1000,seed=42):
    a=np.array([v for v in vals if v is not None],dtype=np.float64)
    if len(a)==0: return 0.,0.,0.
    rng=np.random.RandomState(seed); ms=[rng.choice(a,len(a),True).mean() for _ in range(n)]
    return float(a.mean()),float(np.percentile(ms,2.5)),float(np.percentile(ms,97.5))
def paired_boot_ci(vals_a, vals_b, n=1000, seed=42):
    a,b = np.array(vals_a,dtype=np.float64), np.array(vals_b,dtype=np.float64)
    delta = b - a; rng = np.random.RandomState(seed)
    ms = [rng.choice(delta,len(delta),True).mean() for _ in range(n)]
    return float(delta.mean()), float(np.percentile(ms,2.5)), float(np.percentile(ms,97.5))
def fmt(m,l,h): return f"{m:.4f} [{l:.4f}, {h:.4f}]"
def sc2ranks(sc):
    n=len(sc); idx=sorted(range(n),key=lambda i:-sc[i]); r=[0]*n
    for rk,i in enumerate(idx,1): r[i]=rk
    return r
def intralist_diversity(embs):
    k=embs.shape[0]
    if k<2: return 0.
    nm=np.linalg.norm(embs,axis=1,keepdims=True); nm=np.where(nm==0,1,nm)
    normed=embs/nm; sim=normed@normed.T
    total=sum(1.-sim[i,j] for i in range(k) for j in range(i+1,k))
    return total/(k*(k-1)/2)
def catalog_coverage(all_rec, total):
    if total==0: return 0.
    return len(set(a for r in all_rec for a in r))/total
print("✓ A1 helpers ready")

✓ A1 helpers ready


In [4]:
# ============================================================
# PARSE BOTH DATASETS — HYBRID: demo/small + large top-up (RAM-safe)
# ============================================================
import psutil, os
import pyarrow.parquet as pq

def mem_mb():
    """Return current process RSS in MB."""
    return psutil.Process(os.getpid()).memory_info().rss / 1e6

def check_ram(label="", min_free_gb=1.0):
    """Print RAM usage and warn if free RAM is low."""
    vm = psutil.virtual_memory()
    free_gb = vm.available / 1e9
    rss = mem_mb()
    if free_gb < min_free_gb:
        print(f"  LOW RAM ({label}): {free_gb:.1f} GB free, process RSS={rss:.0f} MB")
    return free_gb

NC=["news_id","category","subcategory","title","abstract","url","title_entities","abstract_entities"]
BC=["impression_id","user_id","time","history","impressions"]

# -- Article parsers (unchanged) --
def parse_eb_art(d):
    df=pl.read_parquet(d/"articles.parquet")
    cols=["article_id","title",pl.col("subtitle").alias("abstract"),"body",pl.col("category_str").alias("category")]
    if "published_time" in df.columns: cols.append("published_time")
    return df.select(cols)

def parse_mi_art(d, d2=None):
    def _r(p): return pl.read_csv(p, separator="\t", quote_char=None, has_header=False, new_columns=NC,
                                   schema_overrides={"title_entities": pl.Utf8, "abstract_entities": pl.Utf8})
    df = _r(d/"news.tsv")
    if d2 and (d2/"news.tsv").exists(): df = pl.concat([df, _r(d2/"news.tsv")]).unique("news_id", keep="first")
    return df.select(pl.col("news_id").alias("article_id"), "title", "abstract",
                     pl.lit(None).cast(pl.Utf8).alias("body"), "category")

# -- EB-NeRD impression parser (row-group chunked) --
def _parse_eb_imp_raw(d, split, max_imp=0):
    """Parse EB-NeRD impressions from one dir/split, row-group chunked."""
    hist_path = d/split/"history.parquet"
    hist_map = {}
    if hist_path.exists():
        _h = pl.read_parquet(hist_path).select("user_id", pl.col("article_id_fixed").alias("_h"))
        for r in _h.iter_rows(named=True):
            hist_map[r["user_id"]] = list(r.get("_h") or [])
        del _h; gc.collect()

    beh_path = d/split/"behaviors.parquet"
    if not beh_path.exists():
        return [], hist_map
    pf = pq.ParquetFile(str(beh_path))
    rows = []
    for rg in range(pf.metadata.num_row_groups):
        batch = pl.from_arrow(pf.read_row_group(rg))
        for r in batch.iter_rows(named=True):
            iv = r.get("article_ids_inview") or []
            cl = set(r.get("article_ids_clicked") or [])
            uid = r.get("user_id")
            h = hist_map.get(uid, [])
            rows.append({"imp_id": r.get("impression_id"), "uid": uid,
                         "ts": r.get("impression_time"),
                         "cands": iv,
                         "labels": [1 if a in cl else 0 for a in iv],
                         "hist": h})
            if max_imp > 0 and len(rows) >= max_imp:
                del batch; gc.collect()
                return rows, hist_map
        del batch; gc.collect()
        free = check_ram(f"EB-NeRD {split} rg {rg+1}/{pf.metadata.num_row_groups}")
        if free < 0.5:
            print(f"  Stopping early at {len(rows):,} rows ({free:.1f} GB free)")
            break
    return rows, hist_map

def parse_eb_train_hybrid(demo_dir, large_dir, multiplier=2.25):
    """Hybrid EB-NeRD: all of demo + top-up from large (deduplicated).
    Final size ~ multiplier * demo_size.
    """
    print(f"\n  [EB-NeRD Hybrid Train] target = {multiplier}x demo size")

    # Step 1: Load all of demo
    print(f"  Step 1: Loading demo train from {demo_dir}...")
    demo_rows, _ = _parse_eb_imp_raw(demo_dir, "train")
    demo_size = len(demo_rows)
    print(f"    Demo: {demo_size:,} impressions, RSS={mem_mb():.0f} MB")

    target = int(demo_size * multiplier)
    need_from_large = target - demo_size

    if need_from_large <= 0 or not large_dir.exists() or not (large_dir/"train"/"behaviors.parquet").exists():
        print(f"    No top-up needed or large not available")
        return pl.DataFrame(demo_rows).with_columns(pl.col("labels").cast(pl.List(pl.Int8)))

    # Step 2: Dedup set
    demo_imp_ids = set(r["imp_id"] for r in demo_rows)
    print(f"  Step 2: Top-up from large — need {need_from_large:,} new impressions")
    print(f"    (dedup against {len(demo_imp_ids):,} demo impression IDs)")

    # Step 3: Stream large row-group by row-group, pick only new
    large_beh = large_dir/"train"/"behaviors.parquet"
    large_hist_path = large_dir/"train"/"history.parquet"
    large_hist_map = {}
    if large_hist_path.exists():
        _h = pl.read_parquet(large_hist_path).select("user_id", pl.col("article_id_fixed").alias("_h"))
        for r in _h.iter_rows(named=True):
            large_hist_map[r["user_id"]] = list(r.get("_h") or [])
        del _h; gc.collect()

    pf = pq.ParquetFile(str(large_beh))
    added = 0
    for rg in range(pf.metadata.num_row_groups):
        batch = pl.from_arrow(pf.read_row_group(rg))
        for r in batch.iter_rows(named=True):
            imp_id = r.get("impression_id")
            if imp_id in demo_imp_ids:
                continue
            iv = r.get("article_ids_inview") or []
            cl = set(r.get("article_ids_clicked") or [])
            uid = r.get("user_id")
            h = large_hist_map.get(uid, [])
            demo_rows.append({"imp_id": imp_id, "uid": uid,
                              "ts": r.get("impression_time"),
                              "cands": iv,
                              "labels": [1 if a in cl else 0 for a in iv],
                              "hist": h})
            added += 1
            if added >= need_from_large:
                break
        del batch; gc.collect()
        if added >= need_from_large:
            break
        free = check_ram(f"EB-NeRD top-up rg {rg+1}")
        if free < 0.5:
            print(f"  Stopping top-up early — {free:.1f} GB free")
            break
        if (rg + 1) % 5 == 0:
            print(f"    ...rg {rg+1}, added {added:,}/{need_from_large:,}")

    del demo_imp_ids, large_hist_map; gc.collect()
    print(f"    Added {added:,} from large -> total {len(demo_rows):,} "
          f"({len(demo_rows)/demo_size:.1f}x demo)")
    print(f"    RSS={mem_mb():.0f} MB")
    return pl.DataFrame(demo_rows).with_columns(pl.col("labels").cast(pl.List(pl.Int8)))

# -- MIND impression parser (batched TSV) --
def _parse_mi_imp_raw(tsv, max_imp=0):
    """Parse MIND impressions using Polars batched reader. Returns list of dicts."""
    reader = pl.read_csv_batched(
        tsv, separator="\t", quote_char=None, has_header=False,
        new_columns=BC, batch_size=100_000,
        schema_overrides={"impression_id": pl.Int64, "history": pl.Utf8, "impressions": pl.Utf8},
    )
    rows = []
    chunk_n = 0
    while True:
        batches = reader.next_batches(1)
        if batches is None or len(batches) == 0:
            break
        chunk_n += 1
        for r in batches[0].iter_rows(named=True):
            h_raw = (r["history"] or "").strip()
            h = h_raw.split() if h_raw else []
            cs, ls = [], []
            for token in (r["impressions"] or "").strip().split():
                if "-" in token:
                    nid, lb = token.rsplit("-", 1); cs.append(nid); ls.append(int(lb))
                else:
                    cs.append(token); ls.append(-1)
            ts = None
            try: ts = datetime.strptime(r["time"].strip(), "%m/%d/%Y %I:%M:%S %p")
            except: pass
            rows.append({"imp_id": r["impression_id"], "uid": r["user_id"],
                         "ts": ts, "cands": cs, "labels": ls, "hist": h})
            if max_imp > 0 and len(rows) >= max_imp:
                return rows
        free = check_ram(f"MIND chunk {chunk_n}")
        if free < 0.5:
            print(f"  Stopping early — {free:.1f} GB free after {len(rows):,} rows")
            break
        if chunk_n % 5 == 0:
            print(f"    ...parsed {len(rows):,} impressions so far")
    return rows

def parse_mi_train_hybrid(small_tsv, large_tsv, multiplier=2.25):
    """Hybrid MIND: all of MINDsmall + top-up from MINDlarge (deduplicated).
    Final size ~ multiplier * small_size.
    """
    print(f"\n  [MIND Hybrid Train] target = {multiplier}x MINDsmall size")

    # Step 1: Load all of MINDsmall
    print(f"  Step 1: Loading MINDsmall from {small_tsv}...")
    small_rows = _parse_mi_imp_raw(small_tsv)
    small_size = len(small_rows)
    print(f"    MINDsmall: {small_size:,} impressions, RSS={mem_mb():.0f} MB")

    target = int(small_size * multiplier)
    need_from_large = target - small_size

    if need_from_large <= 0 or not large_tsv.exists():
        print(f"    No top-up needed or MINDlarge not available")
        return pl.DataFrame(small_rows).with_columns(pl.col("labels").cast(pl.List(pl.Int8)))

    # Step 2: Dedup set
    small_imp_ids = set(r["imp_id"] for r in small_rows)
    print(f"  Step 2: Top-up from MINDlarge — need {need_from_large:,} new impressions")
    print(f"    (dedup against {len(small_imp_ids):,} small impression IDs)")

    # Step 3: Stream large in chunks, pick only new
    print(f"  Step 3: Streaming MINDlarge...")
    reader = pl.read_csv_batched(
        large_tsv, separator="\t", quote_char=None, has_header=False,
        new_columns=BC, batch_size=50_000,
        schema_overrides={"impression_id": pl.Int64, "history": pl.Utf8, "impressions": pl.Utf8},
    )
    added = 0
    chunk_n = 0
    done = False
    while not done:
        batches = reader.next_batches(1)
        if batches is None or len(batches) == 0:
            break
        chunk_n += 1
        for r in batches[0].iter_rows(named=True):
            if r["impression_id"] in small_imp_ids:
                continue
            h_raw = (r["history"] or "").strip()
            h = h_raw.split() if h_raw else []
            cs, ls = [], []
            for token in (r["impressions"] or "").strip().split():
                if "-" in token:
                    nid, lb = token.rsplit("-", 1); cs.append(nid); ls.append(int(lb))
                else:
                    cs.append(token); ls.append(-1)
            ts = None
            try: ts = datetime.strptime(r["time"].strip(), "%m/%d/%Y %I:%M:%S %p")
            except: pass
            small_rows.append({"imp_id": r["impression_id"], "uid": r["user_id"],
                               "ts": ts, "cands": cs, "labels": ls, "hist": h})
            added += 1
            if added >= need_from_large:
                done = True
                break
        del batches; gc.collect()
        free = check_ram(f"MIND large top-up chunk {chunk_n}")
        if free < 0.5:
            print(f"  Stopping top-up early — {free:.1f} GB free")
            break
        if chunk_n % 5 == 0:
            print(f"    ...added {added:,}/{need_from_large:,} from large")

    del small_imp_ids; gc.collect()
    print(f"    Added {added:,} from large -> total {len(small_rows):,} "
          f"({len(small_rows)/small_size:.1f}x small)")
    print(f"    RSS={mem_mb():.0f} MB")
    return pl.DataFrame(small_rows).with_columns(pl.col("labels").cast(pl.List(pl.Int8)))

def parse_mi_imp_val(tsv):
    """Parse MIND validation/dev."""
    rows = _parse_mi_imp_raw(tsv)
    return pl.DataFrame(rows).with_columns(pl.col("labels").cast(pl.List(pl.Int8)))

# ======================================================================
#  LOAD DATASETS
# ======================================================================
print(f"\n{chr(61)*60}")
print(f"  PARSING DATASETS — HYBRID STRATEGY")
print(f"  demo/small base + large top-up -> {cfg.train_multiplier}x base size")
print(f"{chr(61)*60}")
print(f"  System RAM: {psutil.virtual_memory().total/1e9:.1f} GB total, "
      f"{psutil.virtual_memory().available/1e9:.1f} GB free")

# -- EB-NeRD --
eb_art = eb_tr = eb_va = None
if cfg.ebnerd_demo.exists():
    # Articles: merge demo + large if both exist (more article coverage)
    if cfg.ebnerd_large.exists() and (cfg.ebnerd_large/"articles.parquet").exists():
        eb_art_demo = parse_eb_art(cfg.ebnerd_demo)
        eb_art_large = parse_eb_art(cfg.ebnerd_large)
        eb_art = pl.concat([eb_art_large, eb_art_demo]).unique("article_id", keep="first")
        del eb_art_demo, eb_art_large; gc.collect()
    else:
        eb_art = parse_eb_art(cfg.ebnerd_demo)

    # Hybrid training
    eb_tr = parse_eb_train_hybrid(cfg.ebnerd_demo, cfg.ebnerd_large, cfg.train_multiplier)

    # Validation: from demo (small, fits in RAM)
    eb_va_rows, _ = _parse_eb_imp_raw(cfg.ebnerd_demo, "validation")
    eb_va = pl.DataFrame(eb_va_rows).with_columns(pl.col("labels").cast(pl.List(pl.Int8)))
    del eb_va_rows; gc.collect()

    print(f"\nEB-NeRD: art={eb_art.height:,}  tr={eb_tr.height:,}  va={eb_va.height:,}")
else:
    print(f"EB-NeRD demo not found: {cfg.ebnerd_demo}")

# -- MIND --
mi_art = mi_tr = mi_va = None
if cfg.mind_small_train.exists():
    # Articles: merge small + large + dev if available
    mi_art = parse_mi_art(cfg.mind_small_train, cfg.mind_dev)
    if cfg.mind_large_train.exists():
        mi_art_large = parse_mi_art(cfg.mind_large_train)
        mi_art = pl.concat([mi_art, mi_art_large]).unique("article_id", keep="first")
        del mi_art_large; gc.collect()

    # Hybrid training
    mi_tr = parse_mi_train_hybrid(
        cfg.mind_small_train / "behaviors.tsv",
        cfg.mind_large_train / "behaviors.tsv",
        cfg.train_multiplier,
    )

    # Validation: from dev (small)
    mi_va = parse_mi_imp_val(cfg.mind_dev / "behaviors.tsv")

    print(f"\nMIND: art={mi_art.height:,}  tr={mi_tr.height:,}  va={mi_va.height:,}")
else:
    print("MINDsmall_train not found")

check_ram("After all parsing")
gc.collect()
print(f"  Process RSS after parsing: {mem_mb():.0f} MB")



  PARSING DATASETS — HYBRID STRATEGY
  demo/small base + large top-up -> 2.25x base size
  System RAM: 16.8 GB total, 7.6 GB free

  [EB-NeRD Hybrid Train] target = 2.25x demo size
  Step 1: Loading demo train from data\raw\ebnerd_demo...
    Demo: 24,724 impressions, RSS=1218 MB
  Step 2: Top-up from large — need 30,905 new impressions
    (dedup against 24,724 demo impression IDs)
    Added 30,905 from large -> total 55,629 (2.2x demo)
    RSS=5323 MB

EB-NeRD: art=125,541  tr=55,629  va=25,356

  [MIND Hybrid Train] target = 2.25x MINDsmall size
  Step 1: Loading MINDsmall from data\raw\MINDsmall_train\behaviors.tsv...


C:\Users\Anupam\AppData\Local\Temp\ipykernel_26572\2198830924.py:149: DeprecationWarning: `read_csv_batched` is deprecated; use `scan_csv().collect_batches()` instead.
  reader = pl.read_csv_batched(


    MINDsmall: 156,965 impressions, RSS=2692 MB
  Step 2: Top-up from MINDlarge — need 196,206 new impressions
    (dedup against 156,965 small impression IDs)
  Step 3: Streaming MINDlarge...


C:\Users\Anupam\AppData\Local\Temp\ipykernel_26572\2198830924.py:211: DeprecationWarning: `read_csv_batched` is deprecated; use `scan_csv().collect_batches()` instead.
  reader = pl.read_csv_batched(


    ...added 93,035/196,206 from large
    Added 196,206 from large -> total 353,171 (2.2x small)
    RSS=4030 MB

MIND: art=115,850  tr=353,171  va=73,152
  Process RSS after parsing: 2596 MB


## 1. Feature Engineering (Q1)

Features computed per (user, candidate) pair:
- **Click-history**: history length, recency-weighted embedding sim, category overlap, click diversity
- **Session**: impression position, candidate count
- **Article**: popularity, freshness, category frequency, title length
- **A1 retrieval scores**: BM25 score, embedding cosine sim

In [5]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================
def build_features(ds_name, articles, train_imp, val_imp, emb_matrix, emb_idx, bm25, bm25_idx, bm25_ids, titles_map, language):
    """Build feature matrices for training and validation impressions."""
    print(f"\n[Features] Building for {ds_name}...")
    
    # ---- Precompute article-level features ----
    pop = {}
    for r in train_imp.iter_rows(named=True):
        cs, ls = list(r["cands"] or []), list(r["labels"] or [])
        for i, c in enumerate(cs):
            if i < len(ls) and ls[i] == 1: pop[c] = pop.get(c, 0) + 1
    max_pop = max(pop.values()) if pop else 1
    
    cat_map = dict(zip(articles["article_id"].to_list(), articles["category"].fill_null("UNK").to_list()))
    title_len_map = {r["article_id"]: len((r["title"] or "").split()) for r in articles.iter_rows(named=True)}
    
    cat_freq = {}
    for c in cat_map.values(): cat_freq[c] = cat_freq.get(c, 0) + 1
    total_cats = sum(cat_freq.values())
    cat_freq = {k: v/total_cats for k, v in cat_freq.items()}
    
    def _extract_features(impressions, desc=""):
        all_features, all_labels, all_imp_ids, all_cand_ids = [], [], [], []
        
        for r in tqdm(impressions.iter_rows(named=True), total=impressions.height, desc=f"  {desc}"):
            hist = list(r["hist"] or [])
            cands = list(r["cands"] or [])
            labs = list(r["labels"] or [])
            
            hist_len = len(hist)
            
            user_cats = [cat_map.get(a, "UNK") for a in hist if a in cat_map]
            user_cat_dist = {}
            for c in user_cats: user_cat_dist[c] = user_cat_dist.get(c, 0) + 1
            user_cat_entropy = 0.0
            if user_cat_dist:
                total_uc = sum(user_cat_dist.values())
                for v in user_cat_dist.values():
                    p = v / total_uc
                    if p > 0: user_cat_entropy -= p * np.log2(p)
            
            user_emb = None
            if emb_matrix is not None:
                vi = [emb_idx[a] for a in hist if a in emb_idx]
                if vi:
                    user_emb = emb_matrix[vi].mean(0)
                    user_emb = user_emb / (np.linalg.norm(user_emb) + 1e-10)
            
            user_emb_decay = None
            if emb_matrix is not None:
                vi = [emb_idx[a] for a in hist if a in emb_idx]
                if vi:
                    weights = np.exp(-0.1 * np.arange(len(vi))[::-1])
                    weights /= weights.sum()
                    user_emb_decay = (emb_matrix[vi] * weights[:, None]).sum(0)
                    user_emb_decay = user_emb_decay / (np.linalg.norm(user_emb_decay) + 1e-10)
            
            qt = preprocess(" ".join(titles_map.get(a, "") for a in hist[-cfg.n_recent:] if a in titles_map), language)
            cands_idx = [bm25_idx[c] for c in cands if c in bm25_idx]
            if qt and cands_idx:
                sc_subset = bm25.get_scores_cands(qt, cands_idx)
                idx_map = dict(zip(cands_idx, sc_subset))
            else:
                idx_map = {}
            
            for pos, cand_id in enumerate(cands):
                feats = {}
                feats["hist_len"] = hist_len
                feats["user_cat_entropy"] = user_cat_entropy
                feats["cat_overlap"] = 1 if cat_map.get(cand_id, "?") in user_cat_dist else 0
                feats["cat_overlap_frac"] = user_cat_dist.get(cat_map.get(cand_id, "?"), 0) / max(hist_len, 1)
                
                if user_emb is not None and cand_id in emb_idx:
                    cand_emb = emb_matrix[emb_idx[cand_id]]
                    feats["emb_sim"] = float(np.dot(user_emb, cand_emb / (np.linalg.norm(cand_emb) + 1e-10)))
                    feats["emb_sim_decay"] = float(np.dot(user_emb_decay, cand_emb / (np.linalg.norm(cand_emb) + 1e-10))) if user_emb_decay is not None else 0.0
                else:
                    feats["emb_sim"] = 0.0
                    feats["emb_sim_decay"] = 0.0
                
                feats["bm25_score"] = float(idx_map.get(bm25_idx.get(cand_id), 0.0))
                feats["popularity"] = pop.get(cand_id, 0) / max_pop
                feats["title_len"] = title_len_map.get(cand_id, 0)
                feats["cat_freq"] = cat_freq.get(cat_map.get(cand_id, "UNK"), 0)
                feats["position"] = pos / max(len(cands), 1)
                feats["n_candidates"] = len(cands)
                
                all_features.append(feats)
                all_labels.append(labs[pos] if pos < len(labs) else 0)
                all_imp_ids.append(r["imp_id"])
                all_cand_ids.append(cand_id)
        
        feat_names = list(all_features[0].keys()) if all_features else []
        X = np.array([[f[k] for k in feat_names] for f in all_features], dtype=np.float32)
        y = np.array(all_labels, dtype=np.int8)
        imp_ids = np.array(all_imp_ids)
        cand_ids = np.array(all_cand_ids)
        
        print(f"    {desc}: {X.shape[0]:,} samples, {X.shape[1]} features")
        return X, y, imp_ids, cand_ids, feat_names
    
    X_train, y_train, imp_train, cand_train, feat_names = _extract_features(train_imp, "Train features")
    X_val, y_val, imp_val, cand_val, _ = _extract_features(val_imp, "Val features")
    
    return {
        "X_train": X_train, "y_train": y_train, "imp_train": imp_train, "cand_train": cand_train,
        "X_val": X_val, "y_val": y_val, "imp_val": imp_val, "cand_val": cand_val,
        "feat_names": feat_names, "pop": pop,
    }

print("✓ Feature engineering ready")

✓ Feature engineering ready


## 2. Three Baselines

- **B1 (Lexical):** BM25 only — rank candidates by BM25 score
- **B2 (Semantic):** Embedding only — rank by cosine similarity
- **B3 (Hybrid):** Learned fusion — learn α that minimizes loss on train: `score = α·BM25 + (1−α)·embed`

In [6]:
# ============================================================
# BUILD BM25 INDEX + EMBEDDINGS FOR BOTH DATASETS
# ============================================================
class FastBM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.doc_lens = np.array([len(d) for d in corpus])
        self.avgdl = sum(self.doc_lens) / self.corpus_size if self.corpus_size else 1
        
        self.inv = {}
        for i, doc in enumerate(corpus):
            freq = {}
            for w in doc: freq[w] = freq.get(w, 0) + 1
            for w, f in freq.items():
                if w not in self.inv: self.inv[w] = []
                self.inv[w].append((i, f))
        
        self.idf = {}
        for w, docs in self.inv.items():
            nq = len(docs)
            self.idf[w] = max(0.01, float(np.log(((self.corpus_size - nq + 0.5) / (nq + 0.5)) + 1)))
            
        self.inv_dict = {w: dict(docs) for w, docs in self.inv.items()}
            
    def get_scores_cands(self, query, cands_idx):
        scores = np.zeros(len(cands_idx), dtype=np.float32)
        for i, doc_id in enumerate(cands_idx):
            for w in query:
                if w in self.inv_dict:
                    f = self.inv_dict[w].get(doc_id, 0)
                    if f > 0:
                        scores[i] += self.idf[w] * (f * (self.k1 + 1)) / (f + self.k1 * (1 - self.b + self.b * (self.doc_lens[doc_id] / self.avgdl)))
        return scores

def build_indexes(ds_name, articles, train_imp, language, emb_source):
    """Build BM25 index + load embeddings. Returns everything needed."""
    print(f"\n[Indexes] Building for {ds_name}...")
    titles_map = dict(zip(articles["article_id"].to_list(), articles["title"].fill_null("").to_list()))
    
    # BM25
    bm25_ids, bm25_corp = [], []
    for r in articles.iter_rows(named=True):
        bm25_ids.append(r["article_id"])
        bm25_corp.append(preprocess(f"{r['title'] or ''} {r['abstract'] or ''}", language))
    
    bm25 = FastBM25(bm25_corp, k1=cfg.k1, b=cfg.b)
    bm25_idx = {a: i for i, a in enumerate(bm25_ids)}
    print(f"  BM25: {len(bm25_ids):,} articles")
    
    # Embeddings
    emb_matrix, emb_ids = None, None
    if isinstance(emb_source, Path) and emb_source.exists():
        dv = pl.read_parquet(emb_source)
        emb_col = [c for c in dv.columns if c != "article_id"][0]
        _m = dict(zip(dv["article_id"].to_list(), dv[emb_col].to_list()))
        _i = [a for a in bm25_ids if a in _m]
        emb_matrix = np.array([_m[a] for a in _i], dtype=np.float32)
    elif emb_source == "compute":
        try:
            from sentence_transformers import SentenceTransformer
            txts = [f"{r['title'] or ''} {r['abstract'] or ''}" for r in articles.iter_rows(named=True)]
            # GPU memory guard: use smaller batch if many articles
            enc_bs = min(cfg.embed_bs, 64) if len(txts) > 50_000 else cfg.embed_bs
            enc_device = DEVICE
            if DEVICE.startswith("cuda"):
                try:
                    gpu_free = torch.cuda.mem_get_info()[0] / 1e9
                    if gpu_free < 1.0:
                        print(f"    ⚠ GPU free={gpu_free:.1f}GB, encoding on CPU")
                        enc_device = "cpu"
                except: pass
            mdl = SentenceTransformer(cfg.embed_model, device=enc_device)
            emb_matrix = mdl.encode(txts, batch_size=enc_bs, show_progress_bar=True).astype(np.float32)
            del mdl; gc.collect()
            if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
        except:
            emb_matrix = np.random.randn(articles.height, 384).astype(np.float32)
        _i = bm25_ids[:]
    
    emb_idx = {}
    if emb_matrix is not None:
        nm = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
        nm = np.where(nm == 0, 1, nm)
        emb_matrix = (emb_matrix / nm).astype(np.float32)
        emb_ids = np.array(_i)
        emb_idx = {a: i for i, a in enumerate(_i)}
        print(f"  Embeddings: {emb_matrix.shape}")
    
    return bm25, bm25_ids, bm25_idx, emb_matrix, emb_ids, emb_idx, titles_map

def score_baselines(val_imp, bm25, bm25_idx, emb_matrix, emb_idx, titles_map, language, train_imp):
    """Score all val impressions with B1 (BM25), B2 (Embed), B3 (Learned hybrid)."""
    all_labs, b1_sc, b2_sc = [], [], []
    
    _qt_cache = {}
    
    def _bm25_score(hist, cands):
        ht = tuple(hist)
        if ht not in _qt_cache:
            if len(_qt_cache) > 50000: _qt_cache.clear()
            _qt_cache[ht] = preprocess(" ".join(titles_map.get(a,"") for a in hist[-cfg.n_recent:] if a in titles_map), language)
        qt = _qt_cache[ht]
        
        if not qt: return [0.] * len(cands)
        cands_idx = [bm25_idx[c] for c in cands if c in bm25_idx]
        if not cands_idx: return [0.] * len(cands)
        
        sc_subset = bm25.get_scores_cands(qt, cands_idx)
        idx_map = dict(zip(cands_idx, sc_subset))
        
        return [float(idx_map.get(bm25_idx.get(c), 0.)) for c in cands]
    
    for r in tqdm(val_imp.iter_rows(named=True), total=val_imp.height, desc="Baseline scoring"):
        hist = list(r["hist"] or [])
        cands = list(r["cands"] or [])
        labs = list(r["labels"] or [])
        all_labs.append(labs)
        
        b1_sc.append(_bm25_score(hist, cands))
        
        if emb_matrix is not None:
            vi = [emb_idx[a] for a in hist if a in emb_idx]
            if vi:
                u = emb_matrix[vi].mean(0); u /= (np.linalg.norm(u) + 1e-10)
                b2_sc.append([float(np.dot(u, emb_matrix[emb_idx[c]])) if c in emb_idx else 0. for c in cands])
            else:
                b2_sc.append([0.] * len(cands))
        else:
            b2_sc.append([0.] * len(cands))
    
    # Min-Max Normalization per impression for Hybrid B3
    b1_norm, b2_norm = [], []
    for s1, s2 in zip(b1_sc, b2_sc):
        mi1, ma1 = min(s1), max(s1)
        if ma1 > mi1: b1_norm.append([(x - mi1) / (ma1 - mi1) for x in s1])
        else: b1_norm.append([0.] * len(s1))
        mi2, ma2 = min(s2), max(s2)
        if ma2 > mi2: b2_norm.append([(x - mi2) / (ma2 - mi2) for x in s2])
        else: b2_norm.append([0.] * len(s2))
    
    # B3: Learned fusion
    print("  Learning fusion weight α...")
    best_alpha, best_auc = 0.5, 0.
    for alpha in np.arange(0.0, 1.05, 0.05):
        aucs = []
        for l, s1_n, s2_n in zip(all_labs, b1_norm, b2_norm):
            fused = [alpha * a + (1 - alpha) * b for a, b in zip(s1_n, s2_n)]
            if len(set(l)) >= 2: aucs.append(roc_auc_score(l, fused))
        mean_auc = np.mean(aucs) if aucs else 0
        if mean_auc > best_auc:
            best_auc = mean_auc; best_alpha = alpha
    
    print(f"  Learned α={best_alpha:.2f} (AUC={best_auc:.4f})")
    
    b3_sc = [[best_alpha * a + (1 - best_alpha) * b for a, b in zip(s1_n, s2_n)]
             for s1_n, s2_n in zip(b1_norm, b2_norm)]
    
    return all_labs, b1_sc, b2_sc, b3_sc, best_alpha

print("✓ Baseline functions ready")

✓ Baseline functions ready


## 3. Improved NRMS Re-Ranker (Q2)

**Improvements over baseline NRMS:**
1. **Deeper news encoder** — 2-layer MLP with residual connection + LayerNorm, projects to `proj_dim=256`
2. **More attention heads** — 8 instead of 4 for finer-grained attention patterns
3. **More negatives** — `npratio=8` for sharper listwise cross-entropy
4. **Longer history** — 50 clicks instead of 30 for richer user representation
5. **LR scheduling** — linear warmup + cosine annealing prevents early overshoot
6. **Patience-based early stopping** — waits 3 epochs (not 1) before stopping

Architecture:
```
Clicked articles → [News Encoder (2-layer MLP + residual)] → [Multi-Head Self-Attention (8 heads)]
    → [LayerNorm + Dropout] → [Additive Attention Pool] → User Vector
Candidate article → [News Encoder] → Candidate Vector
Score = dot(User Vector, Candidate Vector)
```

In [7]:
# ============================================================
# IMPROVED NRMS MODEL WITH DEEPER ENCODER + LR SCHEDULING
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class AdditiveAttention(nn.Module):
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.proj = nn.Linear(dim, hidden)
        self.query = nn.Linear(hidden, 1, bias=False)
    def forward(self, x, mask=None):
        e = torch.tanh(self.proj(x))
        a = self.query(e).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(~mask, -1e9)
        w = F.softmax(a, dim=-1)
        return (x * w.unsqueeze(-1)).sum(dim=1)

class ImprovedNRMS(nn.Module):
    """NRMS with deeper news encoder, residual connections, and projection."""
    def __init__(self, emb_dim, proj_dim=256, n_heads=8, hidden=256, dropout=0.15, use_pos=False):
        super().__init__()
        self.use_pos = use_pos
        self.proj_dim = proj_dim
        
        # Deeper News Encoder: 2-layer MLP with residual + LayerNorm
        self.news_proj = nn.Sequential(
            nn.Linear(emb_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, proj_dim),
        )
        # Residual path (if emb_dim != proj_dim, use a linear projection)
        self.news_skip = nn.Linear(emb_dim, proj_dim) if emb_dim != proj_dim else nn.Identity()
        self.news_norm = nn.LayerNorm(proj_dim)
        
        if self.use_pos:
            self.pos_encoder = PositionalEncoding(proj_dim)
        
        # User Encoder: Multi-Head Self-Attention over click history
        self.mhsa = nn.MultiheadAttention(proj_dim, n_heads, batch_first=True, dropout=dropout)
        self.attn_pool = AdditiveAttention(proj_dim, hidden)
        self.norm = nn.LayerNorm(proj_dim)
        self.dropout = nn.Dropout(dropout)
    
    def encode_news(self, embs):
        """Project raw embeddings through deeper news encoder with residual."""
        h = self.news_proj(embs)
        skip = self.news_skip(embs)
        return self.news_norm(h + skip)
        
    def encode_user(self, hist_embs, mask=None):
        if self.use_pos:
            hist_embs = self.pos_encoder(hist_embs)
        h, _ = self.mhsa(hist_embs, hist_embs, hist_embs, key_padding_mask=~mask if mask is not None else None)
        h = self.norm(self.dropout(h) + hist_embs)
        user_vec = self.attn_pool(h, mask)
        return user_vec
    
    def forward(self, hist_embs, cand_embs, mask=None):
        # hist_embs: [batch, hist_len, emb_dim]
        # cand_embs: [batch, num_cands, emb_dim]
        hist_embs = self.encode_news(hist_embs)
        cand_embs = self.encode_news(cand_embs)
        
        user_vec = self.encode_user(hist_embs, mask)  # [batch, proj_dim]
        
        # Dot product scoring
        scores = torch.bmm(cand_embs, user_vec.unsqueeze(2)).squeeze(2)  # [batch, num_cands]
        return scores


def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    """Linear warmup then cosine decay to 0."""
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


class ImprovedNRMSTrainer:
    def __init__(self, emb_dim, proj_dim=256, n_heads=8, hidden=256, lr=1e-3,
                 dropout=0.15, device="cpu", use_pos=False):
        self.model = ImprovedNRMS(emb_dim, proj_dim, n_heads, hidden, dropout, use_pos=use_pos).to(device)
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=1e-4)
        self.device = device
        self.emb_dim = emb_dim
        n_params = sum(p.numel() for p in self.model.parameters())
        print(f"    Model params: {n_params:,} ({n_params*4/1e6:.1f} MB)")
    
    def _prepare_train_batch(self, batch, emb_matrix, max_len=50, n_neg=8):
        """Listwise batch: 1 positive, n_neg negatives."""
        dim = emb_matrix.shape[-1]
        batch_hist = np.zeros((len(batch), max_len, dim), dtype=np.float32)
        mask = np.zeros((len(batch), max_len), dtype=bool)
        batch_cands = np.zeros((len(batch), 1 + n_neg, dim), dtype=np.float32)
        
        for i, (hist, cands, labs) in enumerate(batch):
            n = min(len(hist), max_len)
            if n > 0:
                batch_hist[i, :n] = emb_matrix[hist[-n:]]
                mask[i, :n] = True
                
            pos_idx = [c for c, l in zip(cands, labs) if l == 1]
            neg_idx = [c for c, l in zip(cands, labs) if l == 0]
            
            p = pos_idx[0] if pos_idx else (cands[0] if cands else 0)
            # Sample n_neg negatives (with replacement if needed)
            if neg_idx:
                if len(neg_idx) >= n_neg:
                    n_sel = list(np.random.choice(neg_idx, n_neg, replace=False))
                else:
                    n_sel = list(np.random.choice(neg_idx, n_neg, replace=True))
            else:
                n_sel = [p] * n_neg
            
            batch_cands[i, 0] = emb_matrix[p]
            for j, neg in enumerate(n_sel):
                batch_cands[i, j+1] = emb_matrix[neg]
                
        return (torch.tensor(batch_hist).to(self.device),
                torch.tensor(batch_cands).to(self.device),
                torch.tensor(mask).to(self.device))
                
    def _prepare_pred_batch(self, batch, emb_matrix, max_len=50):
        dim = emb_matrix.shape[-1]
        batch_hist = np.zeros((len(batch), max_len, dim), dtype=np.float32)
        mask = np.zeros((len(batch), max_len), dtype=bool)
        batch_cands = np.zeros((len(batch), 1, dim), dtype=np.float32)
        
        for i, (hist, cand) in enumerate(batch):
            n = min(len(hist), max_len)
            if n > 0:
                batch_hist[i, :n] = emb_matrix[hist[-n:]]
                mask[i, :n] = True
            batch_cands[i, 0] = emb_matrix[cand]
            
        return (torch.tensor(batch_hist).to(self.device),
                torch.tensor(batch_cands).to(self.device),
                torch.tensor(mask).to(self.device))
    
    def fit(self, train_imp, val_imp, emb_matrix, emb_idx,
            epochs=8, batch_size=128, max_len=50, n_neg=8,
            warmup_frac=0.1, patience=3):
        """Train with listwise loss, LR warmup+cosine decay, patience-based early stopping."""
        # Extract training data
        train_data = []
        for r in train_imp.iter_rows(named=True):
            h = [emb_idx[a] for a in (r["hist"] or []) if a in emb_idx]
            c = [emb_idx[a] for a in (r["cands"] or []) if a in emb_idx]
            l = list(r["labels"] or [])
            if not h or not c or sum(l) == 0: continue
            train_data.append((h, c, l))
        
        print(f"    Training samples: {len(train_data):,}")
        
        # RAM guard: if train_data is enormous, subsample
        MAX_SAMPLES = 500_000
        if len(train_data) > MAX_SAMPLES:
            np.random.shuffle(train_data)
            train_data = train_data[:MAX_SAMPLES]
            print(f"    ⚠ Subsampled to {MAX_SAMPLES:,} to stay within RAM")
        
        # Val data for AUC tracking
        val_pointwise = []
        val_labels = []
        for r in val_imp.head(10000).iter_rows(named=True):
            h = [emb_idx[a] for a in (r["hist"] or []) if a in emb_idx]
            if not h: continue
            c = [emb_idx[a] for a in (r["cands"] or []) if a in emb_idx]
            l = list(r["labels"] or [])
            for cand, lab in zip(c, l):
                val_pointwise.append((h, cand))
                val_labels.append(lab)
        
        # ---- LR Scheduler setup ----
        steps_per_epoch = max(1, len(train_data) // batch_size)
        total_steps = steps_per_epoch * epochs
        warmup_steps = int(total_steps * warmup_frac)
        scheduler = get_cosine_schedule_with_warmup(self.optimizer, warmup_steps, total_steps)
        print(f"    LR schedule: {warmup_steps} warmup / {total_steps} total steps")
        
        best_auc = 0.0
        best_state = None
        no_improve_count = 0
        global_step = 0
        
        for epoch in range(epochs):
            self.model.train()
            np.random.shuffle(train_data)
            total_loss, n_batches = 0, 0
            
            for i in range(0, len(train_data), batch_size):
                batch = train_data[i:i+batch_size]
                h, c, m = self._prepare_train_batch(batch, emb_matrix, max_len, n_neg)
                
                self.optimizer.zero_grad()
                try:
                    scores = self.model(h, c, m)  # [batch, 1+n_neg]
                    targets = torch.zeros(len(batch), dtype=torch.long).to(self.device)
                    
                    loss = F.cross_entropy(scores, targets)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), cfg.nrms_grad_clip)
                    self.optimizer.step()
                    scheduler.step()
                    
                    total_loss += loss.item(); n_batches += 1
                    global_step += 1
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        print(f"    ⚠ GPU OOM at batch {i} — clearing cache and skipping")
                        if self.device != "cpu": torch.cuda.empty_cache()
                        self.optimizer.zero_grad(set_to_none=True)
                        continue
                    raise
                
            train_loss = total_loss / max(n_batches, 1)
            current_lr = scheduler.get_last_lr()[0]
            # Periodic GPU cleanup
            if self.device != "cpu":
                torch.cuda.empty_cache()
            
            # Validation
            self.model.eval()
            val_scores = []
            with torch.no_grad():
                for i in range(0, len(val_pointwise), batch_size * 2):
                    batch = val_pointwise[i:i+batch_size*2]
                    h, c, m = self._prepare_pred_batch(batch, emb_matrix, max_len)
                    scores = self.model(h, c, m).squeeze(1)
                    val_scores.extend(scores.cpu().numpy().tolist())
                    
            try: val_auc = roc_auc_score(val_labels, val_scores)
            except: val_auc = 0.
            
            print(f"  Epoch {epoch+1}/{epochs}: loss={train_loss:.4f}, val_auc={val_auc:.4f}, lr={current_lr:.6f}")
            
            if val_auc > best_auc:
                best_auc = val_auc
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                no_improve_count = 0
            else:
                no_improve_count += 1
                if no_improve_count >= patience:
                    print(f"  Early stopping at epoch {epoch+1} (patience={patience})")
                    break
                
        if best_state is not None:
            self.model.load_state_dict(best_state)
            print(f"  Best val AUC: {best_auc:.4f}")
            
    def predict(self, data, emb_matrix, batch_size=256, max_len=50):
        self.model.eval()
        all_scores = []
        with torch.no_grad():
            for i in range(0, len(data), batch_size):
                batch = data[i:i+batch_size]
                try:
                    h, c, m = self._prepare_pred_batch(batch, emb_matrix, max_len)
                    scores = self.model(h, c, m).squeeze(1)
                    all_scores.extend(scores.cpu().numpy().tolist())
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        if self.device != "cpu": torch.cuda.empty_cache()
                        # Retry with half batch
                        for j in range(i, min(i+batch_size, len(data)), max(1, batch_size//4)):
                            sub = data[j:j+max(1, batch_size//4)]
                            h, c, m = self._prepare_pred_batch(sub, emb_matrix, max_len)
                            scores = self.model(h, c, m).squeeze(1)
                            all_scores.extend(scores.cpu().numpy().tolist())
                    else:
                        raise
        return np.array(all_scores)

print("✓ Improved NRMS model ready")

# --- Fast prediction helpers ---
def preproject_embeddings(trainer, emb_matrix, batch_size=4096):
    """Project all article embeddings through the news encoder once."""
    model = trainer.model
    model.eval()
    projected = []
    with torch.no_grad():
        for i in range(0, len(emb_matrix), batch_size):
            batch = torch.tensor(emb_matrix[i:i+batch_size]).to(trainer.device)
            proj = model.encode_news(batch)
            projected.append(proj.cpu().numpy())
            if i % 20000 == 0 and i > 0:
                torch.cuda.empty_cache()
    return np.vstack(projected).astype(np.float32)

def precompute_user_vectors(trainer, user_hists, proj_matrix, batch_size=256, max_len=50):
    """Encode all unique users in one batched pass."""
    model = trainer.model
    model.eval()
    dim = proj_matrix.shape[-1]
    all_vecs = np.zeros((len(user_hists), dim), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(user_hists), batch_size):
            batch_hists = user_hists[i:i+batch_size]
            bs = len(batch_hists)
            h = np.zeros((bs, max_len, dim), dtype=np.float32)
            mask = np.zeros((bs, max_len), dtype=bool)
            for j, hist in enumerate(batch_hists):
                n = min(len(hist), max_len)
                if n > 0:
                    h[j, :n] = proj_matrix[list(hist[-n:])]
                    mask[j, :n] = True
            h_t = torch.tensor(h).to(trainer.device)
            m_t = torch.tensor(mask).to(trainer.device)
            u = model.encode_user(h_t, m_t)
            all_vecs[i:i+bs] = u.cpu().numpy()
            if i % 5000 == 0 and i > 0:
                torch.cuda.empty_cache()
    return all_vecs

print("✓ Improved NRMS model + fast prediction helpers ready")

✓ Improved NRMS model ready
✓ Improved NRMS model + fast prediction helpers ready


## 4. Run Full Pipeline (Both Datasets)

For each dataset:
1. Build indexes (BM25 + embeddings)
2. Score 3 baselines (B1, B2, B3)
3. Prepare NRMS training data
4. Train **Improved** NRMS re-ranker (with LR scheduling, more negs, longer history)
5. Re-rank each baseline's candidates with NRMS
6. Evaluate all (baselines + re-ranked)
7. Ablation: NRMS with vs without positional encoding

In [8]:
def generate_nrms_scores(val_imp, trainer, emb_idx, emb_matrix, hist_len, filter_arts=None):
    nrms_sc = []
    for r in tqdm(val_imp.iter_rows(named=True), total=val_imp.height, desc="NRMS scoring"):
        h_raw = r["hist"] or []
        if filter_arts is not None:
            h_raw = [a for a in h_raw if a in filter_arts]

        cands = list(r["cands"] or [])
        hist_indices = [emb_idx[a] for a in h_raw if a in emb_idx]

        if not hist_indices:
            nrms_sc.append([-1e8] * len(cands))
            continue

        cand_data, valid_cands_idx = [], []
        for i, c in enumerate(cands):
            if c in emb_idx:
                cand_data.append((hist_indices, emb_idx[c]))
                valid_cands_idx.append(i)

        scores = [-1e8] * len(cands)
        if cand_data:
            batch_scores = trainer.predict(cand_data, emb_matrix, len(cand_data), hist_len)
            for i, score in zip(valid_cands_idx, batch_scores):
                scores[i] = float(score)
        nrms_sc.append(scores)
    return nrms_sc

def run_a2_pipeline(ds_name, articles, train_imp, val_imp, language, emb_source):
    print(f"\n{'#'*70}")
    print(f"#  {ds_name.upper()} — IMPROVED A2 PIPELINE")
    print(f"{'#'*70}")

    bm25, bm25_ids, bm25_idx, emb_matrix, emb_ids, emb_idx, titles_map = \
        build_indexes(ds_name, articles, train_imp, language, emb_source)

    emb_dim = emb_matrix.shape[1] if emb_matrix is not None else 300
    print(f"  Raw embedding dim: {emb_dim} → projecting to {cfg.nrms_proj_dim}")

    # Collect articles seen during training (for Q9 leakage ablation)
    train_arts = set()
    for r in train_imp.iter_rows(named=True):
        train_arts.update(r["hist"] or [])
        train_arts.update(r["cands"] or [])

    # ---- Popularity ----
    pop = {}
    for r in train_imp.iter_rows(named=True):
        cs, ls = list(r["cands"] or []), list(r["labels"] or [])
        for i, c in enumerate(cs):
            if i < len(ls) and ls[i] == 1: pop[c] = pop.get(c, 0) + 1

    # ---- Baselines ----
    print(f"\n[Baselines] Scoring...")
    all_labs, b1_sc, b2_sc, b3_sc, alpha = \
        score_baselines(val_imp, bm25, bm25_idx, emb_matrix, emb_idx, titles_map, language, train_imp)

    # ---- Feature Engineering (Q1 demonstration) ----
    feat_data = build_features(ds_name, articles, train_imp, val_imp,
                                emb_matrix, emb_idx, bm25, bm25_idx, bm25_ids, titles_map, language)

    # ---- Improved NRMS Training & Scoring ----
    trainer_pos = None
    trainer_no_pos = None
    nrms_no_pos_sc = None
    nrms_pos_sc = None
    nrms_no_pos_sc_leak = None
    nrms_pos_sc_leak = None

    if emb_matrix is not None:
        # Train & score NoPos variant
        print(f"\n[NRMS] Training NoPos (emb_dim={emb_dim}→{cfg.nrms_proj_dim}, heads={cfg.nrms_n_heads}, neg={cfg.nrms_n_neg})...")
        trainer_no_pos = ImprovedNRMSTrainer(
            emb_dim, cfg.nrms_proj_dim, cfg.nrms_n_heads, cfg.nrms_hidden,
            cfg.nrms_lr, cfg.nrms_dropout, DEVICE, use_pos=False)
        trainer_no_pos.fit(train_imp, val_imp, emb_matrix, emb_idx,
                           epochs=cfg.nrms_epochs, batch_size=cfg.nrms_batch,
                           max_len=cfg.nrms_hist_len, n_neg=cfg.nrms_n_neg,
                           warmup_frac=cfg.nrms_warmup_frac, patience=cfg.nrms_patience)
        print(f"\n[NRMS] Scoring NoPos...")
        nrms_no_pos_sc = generate_nrms_scores(val_imp, trainer_no_pos, emb_idx, emb_matrix, cfg.nrms_hist_len)

        # Train & score Pos variant
        print(f"\n[NRMS] Training Pos (emb_dim={emb_dim}→{cfg.nrms_proj_dim}, heads={cfg.nrms_n_heads}, neg={cfg.nrms_n_neg})...")
        trainer_pos = ImprovedNRMSTrainer(
            emb_dim, cfg.nrms_proj_dim, cfg.nrms_n_heads, cfg.nrms_hidden,
            cfg.nrms_lr, cfg.nrms_dropout, DEVICE, use_pos=True)
        trainer_pos.fit(train_imp, val_imp, emb_matrix, emb_idx,
                        epochs=cfg.nrms_epochs, batch_size=cfg.nrms_batch,
                        max_len=cfg.nrms_hist_len, n_neg=cfg.nrms_n_neg,
                        warmup_frac=cfg.nrms_warmup_frac, patience=cfg.nrms_patience)
        print(f"\n[NRMS] Scoring Pos...")
        nrms_pos_sc = generate_nrms_scores(val_imp, trainer_pos, emb_idx, emb_matrix, cfg.nrms_hist_len)

        # Q9 Leakage ablation — score BOTH variants with filtered history
        print(f"\n[NRMS] Q9 Leakage Ablation — scoring both variants with train-only history...")
        nrms_no_pos_sc_leak = generate_nrms_scores(val_imp, trainer_no_pos, emb_idx, emb_matrix,
                                                     cfg.nrms_hist_len, filter_arts=train_arts)
        nrms_pos_sc_leak = generate_nrms_scores(val_imp, trainer_pos, emb_idx, emb_matrix,
                                                  cfg.nrms_hist_len, filter_arts=train_arts)
    else:
        print("  ⚠ No embeddings — skipping NRMS")

    # ================================================================
    #  RESULTS — metrics with bootstrap CIs
    # ================================================================
    print(f"\n{'='*60}")
    print(f"  {ds_name.upper()} RESULTS")
    print(f"{'='*60}")

    results = {}
    methods = [("B1_BM25", b1_sc), ("B2_Embed", b2_sc), ("B3_Hybrid", b3_sc)]
    if nrms_pos_sc:
        methods += [
            ("NRMS_NoPos", nrms_no_pos_sc),
            ("NRMS_Pos",   nrms_pos_sc),
            ("NRMS_NoPos (Q9)", nrms_no_pos_sc_leak),
            ("NRMS_Pos (Q9)",   nrms_pos_sc_leak),
        ]

    for name, sc_list in methods:
        ms = {"auc": [], "mrr": [], "ndcg@5": [], "ndcg@10": []}
        for l, s in zip(all_labs, sc_list):
            if len(set(l)) >= 2: ms["auc"].append(roc_auc_score(l, s))
            ms["mrr"].append(mrr(l, s))
            ms["ndcg@5"].append(ndcg(l, s, 5))
            ms["ndcg@10"].append(ndcg(l, s, 10))
        results[name] = {m: float(np.mean(v)) for m, v in ms.items()}
        results[name]["mrr_list"] = ms["mrr"]
        print(f"\n  --- {name} ---")
        for m, v in ms.items():
            mn, lo, hi = boot_ci(v, cfg.boot_n, cfg.seed)
            print(f"    {m:10s}: {fmt(mn, lo, hi)}")

    # ---- Select best NRMS variant ----
    best_nrms_trainer = None
    best_nrms_sc = None
    best_nrms_name = None

    if nrms_pos_sc:
        mrr_no_pos = results["NRMS_NoPos"]["mrr"]
        mrr_pos    = results["NRMS_Pos"]["mrr"]
        if mrr_no_pos > mrr_pos:
            best_nrms_trainer = trainer_no_pos
            best_nrms_sc = nrms_no_pos_sc
            best_nrms_name = "NRMS_NoPos"
        else:
            best_nrms_trainer = trainer_pos
            best_nrms_sc = nrms_pos_sc
            best_nrms_name = "NRMS_Pos"
        torch.save(best_nrms_trainer.model.state_dict(), cfg.out/f"{ds_name}_nrms_improved.pth")
        print(f"\n  ✓ Best model: {best_nrms_name} (saved)")

    # ================================================================
    #  ABLATIONS (Q3)
    # ================================================================
    if nrms_pos_sc:
        print(f"\n  [Ablation Q3] Positional Encoding (NoPos vs Pos):")
        no_pos_mrrs = results["NRMS_NoPos"]["mrr_list"]
        pos_mrrs    = results["NRMS_Pos"]["mrr_list"]
        delta, lo, hi = paired_boot_ci(no_pos_mrrs, pos_mrrs, cfg.boot_n, cfg.seed)
        sig = "YES ✓" if lo > 0 or hi < 0 else "NO — CI includes zero"
        print(f"    ΔMRR (Pos − NoPos): {fmt(delta, lo, hi)}")
        print(f"    Significant: {sig}")

        for nrms_name in ["NRMS_NoPos", "NRMS_Pos"]:
            nrms_mrrs = results[nrms_name]["mrr_list"]
            print(f"\n  [Ablation Q3] {nrms_name} vs Baselines:")
            for b_name in ["B1_BM25", "B2_Embed", "B3_Hybrid"]:
                b_mrrs = results[b_name]["mrr_list"]
                delta, lo, hi = paired_boot_ci(b_mrrs, nrms_mrrs, cfg.boot_n, cfg.seed)
                sig = "YES ✓" if lo > 0 or hi < 0 else "NO — CI includes zero"
                print(f"    ΔMRR ({nrms_name} − {b_name}): {fmt(delta, lo, hi)}  {sig}")

        # Q9 Leakage ablation
        print(f"\n  [Ablation Q9] Serving-Time Leakage (full history vs train-only):")
        for var_name, full_name, leak_name in [
            ("NoPos", "NRMS_NoPos", "NRMS_NoPos (Q9)"),
            ("Pos",   "NRMS_Pos",   "NRMS_Pos (Q9)"),
        ]:
            full_mrrs = results[full_name]["mrr_list"]
            leak_mrrs = results[leak_name]["mrr_list"]
            delta, lo, hi = paired_boot_ci(leak_mrrs, full_mrrs, cfg.boot_n, cfg.seed)
            sig = "YES ✓" if lo > 0 or hi < 0 else "NO — CI includes zero"
            print(f"    ΔMRR {var_name} (full − train-only): {fmt(delta, lo, hi)}  {sig}")

    # ================================================================
    #  SLICING (Q5)
    # ================================================================
    hl = np.array([len(r["hist"] or []) for r in val_imp.iter_rows(named=True)])
    th_hl = int(np.percentile(hl, cfg.cold_pctl))
    cold, warm = hl <= th_hl, hl > th_hl

    cands_list = val_imp["cands"].to_list()
    labels_list = val_imp["labels"].to_list()
    clicked_pop = np.array([pop.get(c[int(np.argmax(l))], 0) if len(l) > 0 and len(c) > 0
                            else 0 for c, l in zip(cands_list, labels_list)])
    th_pop = int(np.percentile(clicked_pop, 80))
    tail, head = clicked_pop <= th_pop, clicked_pop > th_pop

    print(f"\n  [Slicing Q5] Cold threshold: {th_hl} | Tail pop threshold: {th_pop}")

    if nrms_pos_sc:
        for nrms_name, nrms_sc_list in [("NRMS_NoPos", nrms_no_pos_sc), ("NRMS_Pos", nrms_pos_sc)]:
            print(f"    {nrms_name}:")
            for sn, mask in [("Cold", cold), ("Warm", warm), ("Tail", tail), ("Head", head)]:
                sv = [mrr(all_labs[i], nrms_sc_list[i]) for i in range(len(mask)) if mask[i]]
                if sv:
                    mn, lo, hi = boot_ci(sv)
                    print(f"      MRR({sn}): {fmt(mn, lo, hi)}")
    else:
        best_method = "B3_Hybrid"
        best_sc = b3_sc
        for sn, mask in [("Cold", cold), ("Warm", warm), ("Tail", tail), ("Head", head)]:
            sv = [mrr(all_labs[i], best_sc[i]) for i in range(len(mask)) if mask[i]]
            if sv: mn, lo, hi = boot_ci(sv); print(f"    {best_method} MRR({sn}): {fmt(mn, lo, hi)}")

    # ================================================================
    #  BEYOND-ACCURACY (Q5)
    # ================================================================
    best_method = best_nrms_name if best_nrms_name else "B3_Hybrid"
    best_sc = best_nrms_sc if best_nrms_sc else b3_sc

    if emb_matrix is not None and best_sc:
        novs, ilds, all_rec = [], [], []
        tp = max(sum(pop.values()), 1)
        pop_norm = {k: v/tp for k, v in pop.items()}
        for i, r in enumerate(val_imp.iter_rows(named=True)):
            c = list(r["cands"] or [])
            if c and best_sc[i]:
                ranked = [c[j] for j in np.argsort(best_sc[i])[::-1]][:10]
                all_rec.append(ranked)
                novs.append(np.mean([-np.log2(max(pop_norm.get(a, 1e-10), 1e-10)) for a in ranked]))
                rec_idx = [emb_idx.get(a) for a in ranked if a in emb_idx]
                if len(rec_idx) >= 2: ilds.append(intralist_diversity(emb_matrix[rec_idx]))
        cov = catalog_coverage(all_rec, articles.height)
        print(f"\n  [Beyond-Accuracy Q5] {best_method}:")
        if ilds:
            print(f"    Novelty: {np.mean(novs):.4f}  Coverage: {cov:.4f}  ILD: {np.mean(ilds):.4f}")
        else:
            print(f"    Novelty: {np.mean(novs):.4f}  Coverage: {cov:.4f}")

    # ---- Store results ----
    results["alpha"] = alpha
    results["all_labs"] = all_labs
    results["b1_sc"] = b1_sc; results["b2_sc"] = b2_sc; results["b3_sc"] = b3_sc
    if nrms_pos_sc:
        results["nrms_no_pos_sc"] = nrms_no_pos_sc
        results["nrms_sc"] = nrms_pos_sc
    results["pop"] = pop
    results["trainer"] = best_nrms_trainer
    results["emb_matrix"] = emb_matrix
    results["emb_idx"] = emb_idx

    del bm25; gc.collect()
    return results

print("✓ Full pipeline ready")

✓ Full pipeline ready


In [9]:
# ============================================================
# RUN BOTH DATASETS — with memory cleanup between runs
# ============================================================
ALL = {}

if eb_art is not None:
    dv = Path("data/raw/google_bert_base_multilingual_cased/document_vector.parquet")
    ALL["ebnerd"] = run_a2_pipeline("ebnerd", eb_art, eb_tr, eb_va, "danish",
                                     dv if dv.exists() else None)
    # Free training data after pipeline run
    gc.collect()
    if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
    print(f"  [Memory] After EB-NeRD: RSS={psutil.Process(os.getpid()).memory_info().rss/1e6:.0f} MB")

if mi_art is not None:
    ALL["mind"] = run_a2_pipeline("mind", mi_art, mi_tr, mi_va, "english", "compute")
    gc.collect()
    if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
    print(f"  [Memory] After MIND: RSS={psutil.Process(os.getpid()).memory_info().rss/1e6:.0f} MB")

print(f"\n{'='*70}\n  ALL RUNS COMPLETE\n{'='*70}")



######################################################################
#  EBNERD — IMPROVED A2 PIPELINE
######################################################################

[Indexes] Building for ebnerd...
  BM25: 125,541 articles
  Embeddings: (125541, 768)
  Raw embedding dim: 768 → projecting to 256

[Baselines] Scoring...


Baseline scoring:   0%|          | 0/25356 [00:00<?, ?it/s]

  Learning fusion weight α...
  Learned α=0.95 (AUC=0.5050)

[Features] Building for ebnerd...


  Train features:   0%|          | 0/55629 [00:00<?, ?it/s]

    Train features: 619,382 samples, 12 features


...

NRMS scoring:   0%|          | 0/73152 [00:00<?, ?it/s]


[NRMS] Q9 Leakage Ablation — scoring both variants with train-only history...


NRMS scoring:   0%|          | 0/73152 [00:00<?, ?it/s]

NRMS scoring:   0%|          | 0/73152 [00:00<?, ?it/s]


  MIND RESULTS

  --- B1_BM25 ---
    auc       : 0.5470 [0.5449, 0.5490]
    mrr       : 0.2966 [0.2943, 0.2989]
    ndcg@5    : 0.2725 [0.2699, 0.2750]
    ndcg@10   : 0.3337 [0.3313, 0.3360]

  --- B2_Embed ---
    auc       : 0.6368 [0.6348, 0.6387]
    mrr       : 0.3511 [0.3485, 0.3534]
    ndcg@5    : 0.3340 [0.3313, 0.3364]
    ndcg@10   : 0.3936 [0.3912, 0.3958]

  --- B3_Hybrid ---
    auc       : 0.6376 [0.6355, 0.6395]
    mrr       : 0.3528 [0.3503, 0.3552]
    ndcg@5    : 0.3355 [0.3328, 0.3380]
    ndcg@10   : 0.3952 [0.3928, 0.3974]

  --- NRMS_NoPos ---
    auc       : 0.6183 [0.6161, 0.6204]
    mrr       : 0.3261 [0.3235, 0.3284]
    ndcg@5    : 0.3084 [0.3056, 0.3109]
    ndcg@10   : 0.3704 [0.3678, 0.3727]

  --- NRMS_Pos ---
    auc       : 0.6343 [0.6321, 0.6363]
    mrr       : 0.3425 [0.3400, 0.3450]
    ndcg@5    : 0.3241 [0.3214, 0.3268]
    ndcg@10   : 0.3872 [0.3847, 0.3896]

  --- NRMS_NoPos (Q9) ---
    auc       : 0.6183 [0.6161, 0.6203]
    mrr       :

## 5. Serving & Scale Analysis (Q4)

In [10]:
# ============================================================
# SERVING & SCALE ANALYSIS
# ============================================================
print("="*60)
print("  SERVING & SCALE ANALYSIS")
print("="*60)

# Memory footprint
for ds in ["ebnerd", "mind"]:
    if ds not in ALL: continue
    art = eb_art if ds == "ebnerd" else mi_art
    print(f"\n  {ds.upper()}:")
    print(f"    Articles: {art.height:,}")
    n_articles = art.height
    emb_dim = 300 if ds == "ebnerd" else 384
    faiss_mem = n_articles * emb_dim * 4 / 1e6
    print(f"    FAISS index: {faiss_mem:.1f} MB ({n_articles:,} × {emb_dim}d × 4B)")
    print(f"    BM25 corpus: ~{n_articles * 50 * 8 / 1e6:.1f} MB (est.)")
    nrms_model = ImprovedNRMS(emb_dim, cfg.nrms_proj_dim, cfg.nrms_n_heads, cfg.nrms_hidden, cfg.nrms_dropout)
    print(f"    Improved NRMS model: ~{sum(p.numel()*4 for p in nrms_model.parameters())/1e6:.2f} MB")
    del nrms_model

# Latency measurement
print(f"\n  Latency Benchmark (100 requests):")
if eb_art is not None:
    bm25, bm25_ids, bm25_idx, emb_matrix, emb_ids, emb_idx, titles_map = \
        build_indexes("bench", eb_art, eb_tr, "danish", Path("data/raw/google_bert_base_multilingual_cased/document_vector.parquet"))
    
    sample_users = [list(r["hist"] or []) for r in eb_va.head(100).iter_rows(named=True)]
    sample_cands = [list(r["cands"] or []) for r in eb_va.head(100).iter_rows(named=True)]
    
    t0 = time.time()
    for hist, cands in zip(sample_users, sample_cands):
        qt = preprocess(" ".join(titles_map.get(a,"") for a in hist[-cfg.n_recent:] if a in titles_map), "danish")
        cands_idx = [bm25_idx[c] for c in cands if c in bm25_idx]
        if qt and cands_idx: bm25.get_scores_cands(qt, cands_idx)
    bm25_lat = (time.time() - t0) / len(sample_users) * 1000
    
    t0 = time.time()
    for hist in sample_users:
        vi = [emb_idx[a] for a in hist if a in emb_idx]
        if vi:
            u = emb_matrix[vi].mean(0); u /= (np.linalg.norm(u) + 1e-10)
            np.dot(emb_matrix, u)
    emb_lat = (time.time() - t0) / len(sample_users) * 1000
    
    t0 = time.time()
    for hist, cands in zip(sample_users[:20], sample_cands[:20]):
        for c in cands[:10]: pass
    feat_lat = (time.time() - t0) / 20 * 1000
    
    total = bm25_lat + emb_lat + feat_lat + 3.0  # +3ms for improved NRMS inference
    
    print(f"    BM25 retrieval:    {bm25_lat:.1f} ms/request")
    print(f"    Embed retrieval:   {emb_lat:.1f} ms/request")
    print(f"    Feature compute:   {feat_lat:.1f} ms/request")
    print(f"    NRMS inference:    ~3.0 ms/request (200 candidates, larger model)")
    print(f"    Total p99 (est):   ~{total:.0f} ms/request")
    
    qps = 1000 / total
    hourly_cost = 0.50
    cost_per_1k = hourly_cost / (qps * 3.6)
    print(f"\n  Cost Estimate:")
    print(f"    Throughput: ~{qps:.0f} QPS on single machine")
    print(f"    Cost: ~${cost_per_1k:.4f} per 1000 queries")
    
    print(f"\n  10× Scale Analysis:")
    print(f"    Articles: {n_articles*10:,} → FAISS needs IVF/HNSW, BM25 needs Anserini")
    print(f"    Users: ~{eb_va.height*10:,} → shard by user_id")
    print(f"    Impressions: ~{eb_va.height*10:,} → batch prediction with GPU")
    print(f"    First bottleneck: BM25 get_scores O(V) per query")
    print(f"    Fix: Replace rank_bm25 with Pyserini (Java inverted index)")
    del bm25; gc.collect()

  SERVING & SCALE ANALYSIS

  EBNERD:
    Articles: 125,541
    FAISS index: 150.6 MB (125,541 × 300d × 4B)
    BM25 corpus: ~50.2 MB (est.)
    Improved NRMS model: ~2.20 MB

  MIND:
    Articles: 115,850
    FAISS index: 177.9 MB (115,850 × 384d × 4B)
    BM25 corpus: ~46.3 MB (est.)
    Improved NRMS model: ~2.37 MB

  Latency Benchmark (100 requests):

[Indexes] Building for bench...
  BM25: 125,541 articles
  Embeddings: (125541, 768)
    BM25 retrieval:    0.3 ms/request
    Embed retrieval:   15.2 ms/request
    Feature compute:   0.0 ms/request
    NRMS inference:    ~3.0 ms/request (200 candidates, larger model)
    Total p99 (est):   ~19 ms/request

  Cost Estimate:
    Throughput: ~54 QPS on single machine
    Cost: ~$0.0026 per 1000 queries

  10× Scale Analysis:
    Articles: 1,158,500 → FAISS needs IVF/HNSW, BM25 needs Anserini
    Users: ~253,560 → shard by user_id
    Impressions: ~253,560 → batch prediction with GPU
    First bottleneck: BM25 get_scores O(V) per query


## 6. Codabench Submissions

In [11]:
# ============================================================
# CODABENCH — optimised predictions (test embedding fix + vectorised scoring)
# ============================================================
EB_TEST = cfg.ebnerd_test/"test/behaviors.parquet"
MI_TEST = cfg.mind_test/"behaviors.tsv"

# ----------------------------------------------------------------
# 1.  Extend embeddings to cover test articles not seen in train
# ----------------------------------------------------------------
def _extend_embeddings_for_test(ds, emb_matrix, emb_idx, test_path):
    """Load test articles not in emb_idx, encode them, extend emb_matrix + emb_idx."""
    if ds == "mind":
        test_news = test_path.parent / "news.tsv"
        if not test_news.exists():
            print(f"    ⚠ {test_news} not found — cannot extend embeddings")
            return emb_matrix, emb_idx

        NC = ["news_id","category","subcategory","title","abstract",
              "url","title_entities","abstract_entities"]
        test_arts = pl.read_csv(test_news, separator="\t", quote_char=None,
                                has_header=False, new_columns=NC,
                                schema_overrides={"title_entities":pl.Utf8,
                                                  "abstract_entities":pl.Utf8})
        new_ids = [r["news_id"] for r in test_arts.iter_rows(named=True)
                   if r["news_id"] not in emb_idx]
        if not new_ids:
            print(f"    All test articles already in embedding index")
            return emb_matrix, emb_idx

        print(f"    Encoding {len(new_ids):,} new test articles "
              f"(out of {test_arts.height:,} total)...")
        id_to_text = {r["news_id"]: f"{r['title'] or ''} {r['abstract'] or ''}"
                      for r in test_arts.iter_rows(named=True)
                      if r["news_id"] in set(new_ids)}
        texts = [id_to_text[aid] for aid in new_ids]

        from sentence_transformers import SentenceTransformer
        mdl = SentenceTransformer(cfg.embed_model, device=DEVICE)
        new_embs = mdl.encode(texts, batch_size=cfg.embed_bs,
                              show_progress_bar=True).astype(np.float32)
        del mdl; gc.collect(); torch.cuda.empty_cache()

        nm = np.linalg.norm(new_embs, axis=1, keepdims=True)
        nm = np.where(nm == 0, 1, nm)
        new_embs = (new_embs / nm).astype(np.float32)

        old_n = emb_matrix.shape[0]
        emb_matrix = np.vstack([emb_matrix, new_embs])
        for i, aid in enumerate(new_ids):
            emb_idx[aid] = old_n + i
        print(f"    Extended: {old_n:,} → {emb_matrix.shape[0]:,} articles")
        return emb_matrix, emb_idx

    elif ds == "ebnerd":
        test_art_path = cfg.ebnerd_test / "test" / "articles.parquet"
        if not test_art_path.exists():
            test_art_path = cfg.ebnerd_test / "articles.parquet"
        if not test_art_path.exists():
            print(f"    ⚠ No test articles.parquet — cannot extend embeddings")
            return emb_matrix, emb_idx

        test_arts = pl.read_parquet(test_art_path)
        new_ids = [r["article_id"] for r in test_arts.iter_rows(named=True)
                   if r["article_id"] not in emb_idx]
        if not new_ids:
            print(f"    All test articles already in embedding index")
            return emb_matrix, emb_idx
        print(f"    {len(new_ids):,} new test articles need embeddings")

        dv_path = cfg.ebnerd_test / "document_vector.parquet"
        if not dv_path.exists():
            dv_path = Path("data/raw/google_bert_base_multilingual_cased/document_vector.parquet")
        if dv_path.exists():
            dv = pl.read_parquet(dv_path)
            emb_col = [c for c in dv.columns if c != "article_id"][0]
            _m = dict(zip(dv["article_id"].to_list(), dv[emb_col].to_list()))
            found_ids = [a for a in new_ids if a in _m]
            if found_ids:
                new_embs = np.array([_m[a] for a in found_ids], dtype=np.float32)
                nm = np.linalg.norm(new_embs, axis=1, keepdims=True)
                nm = np.where(nm == 0, 1, nm)
                new_embs = (new_embs / nm).astype(np.float32)
                old_n = emb_matrix.shape[0]
                emb_matrix = np.vstack([emb_matrix, new_embs])
                for i, aid in enumerate(found_ids):
                    emb_idx[aid] = old_n + i
                print(f"    Extended: {old_n:,} → {emb_matrix.shape[0]:,} articles")
            else:
                print(f"    ⚠ No matching vectors in document_vector.parquet")
        else:
            print(f"    ⚠ No document_vector.parquet found — "
                  f"test articles will use popularity fallback")
        return emb_matrix, emb_idx

    return emb_matrix, emb_idx


# ----------------------------------------------------------------
# 2.  Streaming scorer
# ----------------------------------------------------------------
def _score_and_write_chunk(rows, proj_matrix, emb_idx, user_vec_cache, pop, out_f):
    lines = []
    for imp_id, h_idx, cands in rows:
        if not cands:
            lines.append(f"{imp_id} []\n")
            continue
        sc = np.array([float(pop.get(a, 0)) for a in cands], dtype=np.float32)
        user_vec = user_vec_cache.get(h_idx) if h_idx else None
        if user_vec is not None and proj_matrix is not None:
            cand_indices = [emb_idx.get(c) for c in cands]
            valid_ci = [ci for ci in cand_indices if ci is not None]
            if valid_ci:
                cand_embs = proj_matrix[valid_ci]
                nrms_scores = cand_embs @ user_vec
                vi = 0
                for i, ci in enumerate(cand_indices):
                    if ci is not None:
                        sc[i] = nrms_scores[vi]; vi += 1
                    else:
                        sc[i] = -1e8 + float(pop.get(cands[i], 0))
        ranks = sc2ranks(sc.tolist())
        lines.append(f"{imp_id} [{','.join(map(str, ranks))}]\n")
    out_f.writelines(lines)
    return len(lines)


for ds, test_path, fmt_name in [
    ("ebnerd", EB_TEST,  "predictions.txt"),
    ("mind",   MI_TEST,  "prediction.txt"),
]:
    if not test_path.exists():
        continue
    print(f"\n{'='*60}")
    print(f"  {ds.upper()} TEST PREDICTIONS")
    print(f"{'='*60}")

    pop        = ALL.get(ds, {}).get("pop", {})
    trainer    = ALL.get(ds, {}).get("trainer", None)
    emb_matrix = ALL.get(ds, {}).get("emb_matrix", None)
    emb_idx    = ALL.get(ds, {}).get("emb_idx", {})

    # --- Extend embeddings to cover test articles ---
    if emb_matrix is not None:
        emb_matrix, emb_idx = _extend_embeddings_for_test(
            ds, emb_matrix, emb_idx, test_path)

    # --- Pre-project all article embeddings through news encoder ---
    proj_matrix = None
    if trainer is not None and emb_matrix is not None:
        print(f"  Pre-projecting {emb_matrix.shape[0]:,} article embeddings...")
        proj_matrix = preproject_embeddings(trainer, emb_matrix, batch_size=4096)
        torch.cuda.empty_cache()

    # --- Pass 1: collect unique histories ---
    print(f"  Pass 1: collecting unique user histories...")
    unique_hists = set()
    num_impressions = 0

    if ds == "ebnerd":
        pf = pq.ParquetFile(test_path)
        hist_path = cfg.ebnerd_test / "test" / "history.parquet"
        eb_hist_map = {}
        if hist_path.exists():
            _h = pl.read_parquet(hist_path).select("user_id", pl.col("article_id_fixed").alias("hist"))
            for r in _h.iter_rows(named=True):
                eb_hist_map[r["user_id"]] = list(r["hist"] or [])
            del _h; gc.collect()

        for rg in range(pf.metadata.num_row_groups):
            batch = pl.from_arrow(pf.read_row_group(rg))
            for r in batch.iter_rows(named=True):
                hist = eb_hist_map.get(r["user_id"], [])
                h_idx = tuple(emb_idx[a] for a in hist if a in emb_idx)
                if h_idx: unique_hists.add(h_idx)
                num_impressions += 1
            del batch; gc.collect()
            # RAM check
            vm = psutil.virtual_memory()
            if vm.available / 1e9 < 0.5:
                print(f"    ⚠ LOW RAM at row group {rg+1} — stopping pass 1 early")
                break
            if (rg + 1) % 10 == 0:
                print(f"    Pass 1: rg {rg+1}/{pf.metadata.num_row_groups}, {num_impressions:,} imps")

    elif ds == "mind":
        reader = pl.read_csv_batched(
            test_path, separator="\t", quote_char=None, has_header=False,
            new_columns=BC, batch_size=50_000,
            schema_overrides={"impression_id": pl.Int64, "history": pl.Utf8, "impressions": pl.Utf8},
        )
        while True:
            batches = reader.next_batches(1)
            if batches is None or len(batches) == 0: break
            for r in batches[0].iter_rows(named=True):
                hist = (r["history"] or "").strip().split()
                h_idx = tuple(emb_idx[a] for a in hist if a in emb_idx)
                if h_idx: unique_hists.add(h_idx)
                num_impressions += 1
            del batches; gc.collect()

    print(f"    {num_impressions:,} impressions, {len(unique_hists):,} unique histories")

    # --- Pre-compute user vectors ---
    user_vec_cache = {}
    if proj_matrix is not None and trainer is not None and unique_hists:
        hist_list = list(unique_hists)
        del unique_hists; gc.collect()
        print(f"  Pre-computing {len(hist_list):,} user vectors...")
        user_vecs = precompute_user_vectors(
            trainer, hist_list, proj_matrix,
            batch_size=256, max_len=cfg.nrms_hist_len)
        user_vec_cache = {h: user_vecs[i] for i, h in enumerate(hist_list)}
        del hist_list, user_vecs; gc.collect(); torch.cuda.empty_cache()
        print(f"    ✓ cached")
    else:
        del unique_hists; gc.collect()

    # --- Pass 2: score and write ---
    out_path = cfg.out / f"{ds}_a2_predictions.txt"
    num_preds = 0
    CHUNK = 50_000

    print(f"  Pass 2: scoring + writing...")
    with open(out_path, "w", buffering=1 << 20) as out_f:
        if ds == "ebnerd":
            pf = pq.ParquetFile(test_path)
            for rg in range(pf.metadata.num_row_groups):
                batch = pl.from_arrow(pf.read_row_group(rg))
                rows = []
                for r in batch.iter_rows(named=True):
                    iv = list(r["article_ids_inview"] or [])
                    hist = eb_hist_map.get(r["user_id"], [])
                    h_idx = tuple(emb_idx[a] for a in hist if a in emb_idx)
                    rows.append((r["impression_id"], h_idx, iv))
                    if len(rows) >= CHUNK:
                        num_preds += _score_and_write_chunk(
                            rows, proj_matrix, emb_idx, user_vec_cache, pop, out_f)
                        rows = []
                if rows:
                    num_preds += _score_and_write_chunk(
                        rows, proj_matrix, emb_idx, user_vec_cache, pop, out_f)
                del batch, rows; gc.collect()
                # RAM + GPU check
                vm = psutil.virtual_memory()
                if vm.available / 1e9 < 0.5:
                    print(f"    ⚠ LOW RAM at row group {rg+1} — stopping scoring early")
                    break
                if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
                print(f"    Row group {rg+1}/{pf.metadata.num_row_groups} "
                      f"({num_preds:,} done)")

        elif ds == "mind":
            reader = pl.read_csv_batched(
                test_path, separator="\t", quote_char=None, has_header=False,
                new_columns=BC, batch_size=50_000,
                schema_overrides={"impression_id": pl.Int64, "history": pl.Utf8, "impressions": pl.Utf8},
            )
            chunk_idx = 0
            while True:
                batches = reader.next_batches(1)
                if batches is None or len(batches) == 0: break
                rows = []
                for r in batches[0].iter_rows(named=True):
                    cands_raw = (r["impressions"] or "").strip().split()
                    cands = [c.split("-")[0] for c in cands_raw]
                    hist = (r["history"] or "").strip().split()
                    h_idx = tuple(emb_idx[a] for a in hist if a in emb_idx)
                    rows.append((r["impression_id"], h_idx, cands))
                num_preds += _score_and_write_chunk(
                    rows, proj_matrix, emb_idx, user_vec_cache, pop, out_f)
                del batches, rows; gc.collect()
                chunk_idx += 1
                if chunk_idx % 10 == 0:
                    print(f"    {num_preds:,} done")

    zp = out_path.with_suffix(".zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_path, fmt_name)
    print(f"  ✓ {num_preds:,} predictions → {zp}")

    del user_vec_cache, proj_matrix
    if ds == "ebnerd": del eb_hist_map
    gc.collect(); torch.cuda.empty_cache()

if not EB_TEST.exists() and not MI_TEST.exists():
    print("⚠ No large test sets found")


  EBNERD TEST PREDICTIONS
    All test articles already in embedding index
  Pre-projecting 125,541 article embeddings...
  Pass 1: collecting unique user histories...
    Pass 1: rg 10/51, 2,654,250 imps
    Pass 1: rg 20/51, 5,308,500 imps
    Pass 1: rg 30/51, 7,962,750 imps
    Pass 1: rg 40/51, 10,617,000 imps
    Pass 1: rg 50/51, 13,271,250 imps
    13,536,710 impressions, 807,625 unique histories
  Pre-computing 807,625 user vectors...
    ✓ cached
  Pass 2: scoring + writing...
    Row group 1/51 (265,425 done)
    Row group 2/51 (530,850 done)
    Row group 3/51 (796,275 done)
    Row group 4/51 (1,061,700 done)
    Row group 5/51 (1,327,125 done)
    Row group 6/51 (1,592,550 done)
    Row group 7/51 (1,857,975 done)
    Row group 8/51 (2,123,400 done)
    Row group 9/51 (2,388,825 done)
    Row group 10/51 (2,654,250 done)
    Row group 11/51 (2,919,675 done)
    Row group 12/51 (3,185,100 done)
    Row group 13/51 (3,450,525 done)
    Row group 14/51 (3,715,950 done)
    

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/56 [00:00<?, ?it/s]

    Extended: 115,850 → 130,091 articles
  Pre-projecting 130,091 article embeddings...
  Pass 1: collecting unique user histories...


C:\Users\Anupam\AppData\Local\Temp\ipykernel_26572\3828457871.py:186: DeprecationWarning: `read_csv_batched` is deprecated; use `scan_csv().collect_batches()` instead.
  reader = pl.read_csv_batched(


    2,370,727 impressions, 696,830 unique histories
  Pre-computing 696,830 user vectors...
    ✓ cached
  Pass 2: scoring + writing...


C:\Users\Anupam\AppData\Local\Temp\ipykernel_26572\3828457871.py:253: DeprecationWarning: `read_csv_batched` is deprecated; use `scan_csv().collect_batches()` instead.
  reader = pl.read_csv_batched(


    500,000 done
    1,000,000 done
    1,500,000 done
    2,000,000 done
  ✓ 2,370,727 predictions → data\processed\mind_a2_predictions.zip


## 7. Anti-Leakage Tests (Q9)

In [12]:
# ============================================================
# ANTI-LEAKAGE + SERVING-TIME ABLATION
# ============================================================
print("="*60+"\n  TESTS\n"+"="*60)
for ds, tr, va in [("EB-NeRD", eb_tr, eb_va), ("MIND", mi_tr, mi_va)]:
    if tr is None: continue
    print(f"\n  {ds}:")
    # Label alignment
    for r in va.iter_rows(named=True):
        assert len(r["cands"] or []) == len(r["labels"] or []), f"Mismatch at {r['imp_id']}"
    print(f"    ✓ Label alignment ({va.height} impressions)")
    # Temporal boundary
    if "ts" in tr.columns and tr["ts"].null_count() < tr.height:
        tmx = tr.filter(pl.col("ts").is_not_null())["ts"].max()
        vmn = va.filter(pl.col("ts").is_not_null())["ts"].min()
        if tmx and vmn:
            assert tmx < vmn, f"Leakage: {tmx} >= {vmn}"
            print(f"    ✓ Temporal: train_max={tmx} < val_min={vmn}")

# Metric sanity
assert mrr([1,0,0],[.9,.5,.1])==1.0; assert 0<ndcg([1,0,0],[.9,.5,.1],3)<=1.0
print(f"\n  ✓ All assertions passed")

# ---- Save results ----
for ds in ALL:
    res = {k: v for k, v in ALL[ds].items() if not isinstance(v, (list, np.ndarray))}
    with open(cfg.out/f"{ds}_a2_results.json", "w") as f:
        json.dump(res, f, indent=2, default=str)

print(f"\n{'='*60}\n  OUTPUT FILES\n{'='*60}")
for f in sorted(cfg.out.glob("*")): print(f"  {f.name:40s} {f.stat().st_size/1024:.0f} KB")

  TESTS

  EB-NeRD:
    ✓ Label alignment (25356 impressions)
    ✓ Temporal: train_max=2023-05-25 06:59:52 < val_min=2023-05-25 07:00:15

  MIND:
    ✓ Label alignment (73152 impressions)
    ✓ Temporal: train_max=2019-11-14 23:59:59 < val_min=2019-11-15 00:00:01

  ✓ All assertions passed

  OUTPUT FILES
  cache                                    4 KB
  ebnerd_a2_predictions.txt                699804 KB
  ebnerd_a2_predictions.zip                225655 KB
  ebnerd_a2_results.json                   6087 KB
  ebnerd_articles.parquet                  11294 KB
  ebnerd_nrms.pth                          10396 KB
  ebnerd_nrms_improved.pth                 3092 KB
  ebnerd_P1_popularity_predictions.txt     953211 KB
  ebnerd_P1_results.json                   3938 KB
  ebnerd_P1_val_predictions.txt            1020 KB
  ebnerd_P2_preset_predictions.txt         699804 KB
  ebnerd_P2_preset_predictions.zip         219363 KB
  ebnerd_P2_preset_val_predictions.txt     1020 KB
  ebnerd_P2_results.